# Hammurab.AI — Turkish Legal GPT-2 Fine-Tuning
**CIF425 Term Project**

Bu notebook Türkçe GPT-2 modelini hukuk Q&A verisiyle fine-tune eder.

**Kullanım:** Runtime > Change runtime type > **T4 GPU** seç, sonra **Run All**

## 1. Repo'yu Klonla & Bağımlılıkları Kur

In [ ]:
!git clone https://github.com/Alp33er/hammurab.ai.git
%cd hammurab.ai
!git checkout development

In [ ]:
!pip install -q transformers torch datasets accelerate sentence-transformers

## 2. GPU Kontrol

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3. Veri Seti Oluştur
Kanun JSON'larından 18K+ Q&A çifti üretir.

In [ ]:
!python training/prepare_dataset.py

In [ ]:
# Veri seti örneği
import json
with open("training/data/hukuk_qa.jsonl", "r") as f:
    sample = json.loads(f.readline())
print("PROMPT:")
print(sample["prompt"][:300])
print("\nCOMPLETION:")
print(sample["completion"][:300])

## 4. Fine-Tuning
~30-45 dakika (T4 GPU)

In [ ]:
!python training/fine_tune.py

## 5. Modeli Test Et

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("./model")
model = AutoModelForCausalLM.from_pretrained("./model").to("cuda")
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def sor(soru, mevzuat=""):
    prompt = f"### Kullanıcı:\n{soru}"
    if mevzuat:
        prompt += f"\n\n### Mevzuat:\n{mevzuat}"
    prompt += "\n\n### Asistan:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    for stop in ["### Kullanıcı", "### Sistem", "### Mevzuat"]:
        if stop in response:
            response = response[:response.index(stop)]
    return response.strip()

print("Model hazır! Aşağıdan test edin.")

In [ ]:
# Test 1: Genel soru
print(sor("İş kazası durumunda işçinin hakları nelerdir?"))

In [ ]:
# Test 2: Belirli madde
print(sor("Türk Borçlar Kanunu madde 49 ne diyor?"))

In [ ]:
# Test 3: Kendi sorunuzu yazın
print(sor("Kıdem tazminatı nasıl hesaplanır?"))

## 6. Modeli İndir
Eğitilmiş modeli zip'leyip indirebilirsin.

In [ ]:
!zip -r /content/hammurab_model.zip model/

from google.colab import files
files.download("/content/hammurab_model.zip")

## 7. (Alternatif) Modeli Google Drive'a Kaydet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r model/ /content/drive/MyDrive/hammurab_model/
print("Model Google Drive'a kaydedildi!")